In [4]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

HITTER_CSV = Path("hitters_cleaned.csv")

hitters = pd.read_csv(HITTER_CSV, low_memory=False)
print(f"Loaded {len(hitters):,} rows, {len(hitters.columns)} columns")

inf = hitters[hitters["resolved_position"].isin(["SS", "2B", "3B", "1B"])]
inf["resolved_position"].value_counts()

Loaded 32,999 rows, 50 columns


resolved_position
SS    7951
3B    3644
1B    3047
2B    2226
Name: count, dtype: int64

In [5]:
INF_MODELING_COLS = [
    # Identity / bio
    "player_state", "resolved_position",
    "height", "weight", "throwing_hand", "hitting_handedness",
    # Hitting stats (value + date for each)
    "exit_velo_max", "exit_velo_max_date",
    "exit_velo_avg", "exit_velo_avg_date",
    "distance_max", "distance_max_date",
    "sweet_spot_p", "sweet_spot_p_date",
    "hand_speed_max", "hand_speed_max_date",
    "bat_speed_max", "bat_speed_max_date",
    "rot_acc_max", "rot_acc_max_date",
    "hard_hit_p", "hard_hit_p_date",
    # Running and Defense (value + date)
    "sixty_time", "sixty_time_date",
    "thirty_time", "thirty_time_date",
    "ten_yard_time", "ten_yard_time_date",
    "run_speed_max", "run_speed_max_date",
    "inf_velo", "inf_velo_date",
    # Classification
    "player_region", "committment_group", "commitment_date"
]

INF_MODELING_COLS = [c for c in INF_MODELING_COLS if c in inf.columns]
inf_modeling = inf[INF_MODELING_COLS]

inf_modeling.shape

(16868, 35)

In [7]:
# unknown is Non D1, with random juco/naia or canadian colleges or malformed urls for schools.
inf_modeling = inf_modeling[inf_modeling["committment_group"] != "Unknown"] # removing unknown 1000ish rows

print(inf_modeling['committment_group'].value_counts())
print(inf_modeling.shape)

# new target feature for d1 or not modeling
inf_modeling['d1_or_not'] = (inf_modeling['committment_group'] != 'Non D1').astype(int)

inf_modeling['d1_or_not'].value_counts()

committment_group
Non D1       11363
Non P4 D1     3224
P4             864
Name: count, dtype: int64
(15451, 35)


d1_or_not
0    11363
1     4088
Name: count, dtype: int64

In [ ]:
# ============================================================
# STALE DATA ANALYSIS
# ============================================================
# ---- EDITABLE PARAMETERS (change & re-run just this cell) ----
STALE_MONTHS = 24            # Stats older than this many months before graduation = "stale"
OUTLIER_STD_DEV = 2        # ± this many std devs from commitment-group mean = "outlier"
# ---------------------------------------------------------------

# --- Fresh copy so of_modeling stays untouched ---
inf_model_recent = inf_modeling.copy()

# --- Bring in class year (not in inF_MODELING_COLS) ---
inf_model_recent["class"] = inf.loc[inf_model_recent.index, "class"]

# --- Stat <-> date column pairs (only OF-relevant stats) ---
STAT_DATE_PAIRS = [
    ("exit_velo_max",  "exit_velo_max_date"),
    ("exit_velo_avg",  "exit_velo_avg_date"),
    ("distance_max",   "distance_max_date"),
    ("sweet_spot_p",   "sweet_spot_p_date"),
    ("hand_speed_max", "hand_speed_max_date"),
    ("bat_speed_max",  "bat_speed_max_date"),
    ("rot_acc_max",    "rot_acc_max_date"),
    ("hard_hit_p",     "hard_hit_p_date"),
    ("sixty_time",     "sixty_time_date"),
    ("thirty_time",    "thirty_time_date"),
    ("ten_yard_time",  "ten_yard_time_date"),
    ("run_speed_max",  "run_speed_max_date"),
    ("inf_velo",       "inf_velo_date"),
]
# Filter to columns that actually exist
STAT_DATE_PAIRS = [(s, d) for s, d in STAT_DATE_PAIRS if s in inf_model_recent.columns and d in inf_model_recent.columns]

# --- Parse dates & compute months before graduation ---
def parse_pbr_date(d):
    """Parse PBR date format M/DD/YY to datetime."""
    if pd.isna(d) or str(d).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(d).strip(), format="%m/%d/%y")
    except Exception:
        try:
            return pd.to_datetime(str(d).strip())
        except Exception:
            return pd.NaT

# Graduation reference = June 1 of class year
inf_model_recent["grad_date"] = pd.to_datetime(inf_model_recent["class"].astype(str) + "-06-01")

for stat_col, date_col in STAT_DATE_PAIRS:
    parsed_col = f"_{date_col}_parsed"
    months_col = f"{stat_col}__months_before_grad"
    inf_model_recent[parsed_col] = inf_model_recent[date_col].apply(parse_pbr_date)
    # Positive = stat recorded BEFORE graduation, negative = after graduation
    inf_model_recent[months_col] = ((inf_model_recent["grad_date"] - inf_model_recent[parsed_col]).dt.days / 30.44).round(1)

print(f"Parsed {len(STAT_DATE_PAIRS)} stat date columns.\n")

# ============================================================
# 1. PER-STAT STALENESS BREAKDOWN
# ============================================================
print(f"{'='*70}")
print(f"1. PER-STAT STALENESS  (stale = >{STALE_MONTHS} months before graduation)")
print(f"{'='*70}\n")

rows = []
for stat_col, date_col in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    has_val  = inf_model_recent[stat_col].notna()
    has_date = inf_model_recent[months_col].notna()
    is_stale = inf_model_recent[months_col] > STALE_MONTHS

    n_val   = has_val.sum()
    n_dated = (has_val & has_date).sum()
    n_stale = (has_val & is_stale).sum()
    n_no_date = n_val - n_dated

    rows.append({
        "stat": stat_col,
        "has_value": n_val,
        "has_date": n_dated,
        "no_date": n_no_date,
        "stale": n_stale,
        "stale_pct": round(100 * n_stale / n_val, 1) if n_val else 0,
    })

stale_df = pd.DataFrame(rows)
print(stale_df.to_string(index=False))

total_vals  = stale_df["has_value"].sum()
total_stale = stale_df["stale"].sum()
print(f"\nOverall: {total_stale:,} / {total_vals:,} stat values are stale ({100*total_stale/total_vals:.1f}%)")

# ============================================================
# 2. SENSITIVITY — staleness at different month thresholds
# ============================================================
print(f"\n{'='*70}")
print(f"2. STALENESS AT DIFFERENT THRESHOLDS")
print(f"{'='*70}\n")

thresholds = [12, 18, 24, 30, 36, 48]
for thresh in thresholds:
    t_stale = 0
    t_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val  = inf_model_recent[stat_col].notna()
        is_stale = inf_model_recent[months_col] > thresh
        t_stale += (has_val & is_stale).sum()
        t_total += has_val.sum()
    marker = "  <-- current" if thresh == STALE_MONTHS else ""
    print(f"  {thresh:>2} months: {t_stale:>5,} / {t_total:>6,} values stale ({100*t_stale/t_total:.1f}%){marker}")

# ============================================================
# 3. STALENESS BY COMMITMENT GROUP
# ============================================================
print(f"\n{'='*70}")
print(f"3. STALENESS BY COMMITMENT GROUP  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = inf_model_recent["committment_group"] == group
    g_stale = 0
    g_total = 0
    for stat_col, _ in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        has_val  = inf_model_recent.loc[mask, stat_col].notna()
        is_stale = inf_model_recent.loc[mask, months_col] > STALE_MONTHS
        g_stale += (has_val & is_stale).sum()
        g_total += has_val.sum()
    pct = 100 * g_stale / g_total if g_total else 0
    n_players = mask.sum()
    print(f"  {group:>12}  ({n_players:>5,} players): {g_stale:>4,} / {g_total:>5,} values stale ({pct:.1f}%)")

# ============================================================
# 4. PLAYERS WITH ANY STALE STAT
# ============================================================
print(f"\n{'='*70}")
print(f"4. PLAYERS WITH ≥1 STALE STAT  (threshold = {STALE_MONTHS} months)")
print(f"{'='*70}\n")

# For each player, count how many of their stats are stale
stale_counts_per_player = pd.Series(0, index=inf_model_recent.index)
stat_counts_per_player  = pd.Series(0, index=inf_model_recent.index)

for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    has_val  = inf_model_recent[stat_col].notna()
    is_stale = inf_model_recent[months_col] > STALE_MONTHS
    stale_counts_per_player += (has_val & is_stale).astype(int)
    stat_counts_per_player  += has_val.astype(int)

inf_model_recent["_n_stale_stats"] = stale_counts_per_player
inf_model_recent["_n_total_stats"] = stat_counts_per_player

has_any_stale = stale_counts_per_player > 0
for group in ["P4", "Non P4 D1", "Non D1"]:
    mask = inf_model_recent["committment_group"] == group
    n_total = mask.sum()
    n_affected = (mask & has_any_stale).sum()
    print(f"  {group:>12}: {n_affected:>4,} / {n_total:>5,} players have ≥1 stale stat ({100*n_affected/n_total:.1f}%)")

total_affected = has_any_stale.sum()
print(f"  {'Total':>12}: {total_affected:>4,} / {len(inf_model_recent):>5,} players ({100*total_affected/len(inf_model_recent):.1f}%)")

# ============================================================
# 5. OUTLIER + STALENESS INTERSECTION
# ============================================================
print(f"\n{'='*70}")
print(f"5. OUTLIER ANALYSIS  (±{OUTLIER_STD_DEV} std dev from group mean)")
print(f"   → How many outliers have stale stats?")
print(f"{'='*70}\n")

stat_value_cols = [s for s, _ in STAT_DATE_PAIRS]
outlier_rows = []

for stat_col in stat_value_cols:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = inf_model_recent["committment_group"] == group
        vals = inf_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std  = vals.std()

        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue

        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        n_outlier  = is_outlier.sum()
        n_with_val = vals.notna().sum()

        # Of those outliers, how many are stale?
        outlier_stale = (is_outlier & (inf_model_recent.loc[mask, months_col] > STALE_MONTHS)).sum()
        # And how many are fresh?
        outlier_fresh = n_outlier - outlier_stale

        outlier_rows.append({
            "stat": stat_col, "group": group,
            "n": n_with_val, "outliers": n_outlier,
            "outlier_pct": round(100 * n_outlier / n_with_val, 1) if n_with_val else 0,
            "stale_outliers": outlier_stale,
            "fresh_outliers": outlier_fresh,
            "pct_outliers_stale": round(100 * outlier_stale / n_outlier, 1) if n_outlier else 0,
        })

outlier_df = pd.DataFrame(outlier_rows)

# Summary table: aggregate across stats per group
print("Aggregated across all stats per commitment group:\n")
for group in ["P4", "Non P4 D1", "Non D1"]:
    sub = outlier_df[outlier_df["group"] == group]
    tot_outliers = sub["outliers"].sum()
    tot_stale_o  = sub["stale_outliers"].sum()
    tot_fresh_o  = sub["fresh_outliers"].sum()
    pct = 100 * tot_stale_o / tot_outliers if tot_outliers else 0
    print(f"  {group:>12}: {tot_outliers:>4} outliers total → {tot_stale_o:>3} stale ({pct:.0f}%), {tot_fresh_o:>3} fresh ({100-pct:.0f}%)")

print(f"\nFull per-stat breakdown:")
print(outlier_df.to_string(index=False))

# ============================================================
# 6. SUMMARY & DECISION HELPER
# ============================================================
print(f"\n{'='*70}")
print(f"6. IMPACT SUMMARY — What happens if we NaN stale stats?")
print(f"{'='*70}\n")

# Option A: NaN ALL stale stats
print(f"Option A — NaN ALL stats older than {STALE_MONTHS} months:")
remaining_a = total_vals - total_stale
print(f"  Stat values removed: {total_stale:,} / {total_vals:,} ({100*total_stale/total_vals:.1f}%)")
print(f"  Stat values remaining: {remaining_a:,}")

# How many players would lose ALL their stats?
all_stale = (stale_counts_per_player == stat_counts_per_player) & (stat_counts_per_player > 0)
print(f"  Players losing ALL stats: {all_stale.sum():,} / {len(inf_model_recent):,}")

# Option B: NaN only stale OUTLIER stats
total_stale_outliers = outlier_df["stale_outliers"].sum()
print(f"\nOption B — NaN only stale outliers (±{OUTLIER_STD_DEV} std + >{STALE_MONTHS}mo):")
print(f"  Stat values removed: {total_stale_outliers:,} / {total_vals:,} ({100*total_stale_outliers/total_vals:.1f}%)")
print(f"  More surgical, but leaves non-outlier stale data in place")

# Clean up internal columns from inf_model_recent (keep months_before_grad cols for downstream use)
internal_cols = [c for c in inf_model_recent.columns if c.startswith("_")]
inf_model_recent.drop(columns=internal_cols, inplace=True)

Parsed 13 stat date columns.

1. PER-STAT STALENESS  (stale = >24 months before graduation)

          stat  has_value  has_date  no_date  stale  stale_pct
 exit_velo_max      12764     12764        0   2162       16.9
 exit_velo_avg      11410     11410        0   1917       16.8
  distance_max      11409     11409        0   1950       17.1
  sweet_spot_p      11386     11386        0   3363       29.5
hand_speed_max      10609     10609        0   2594       24.5
 bat_speed_max      10609     10609        0   2070       19.5
   rot_acc_max      10609     10609        0   2566       24.2
    hard_hit_p       7673      7673        0    847       11.0
    sixty_time      12100     12100        0   2453       20.3
   thirty_time       5688      5688        0    689       12.1
 ten_yard_time       5977      5977        0    864       14.5
 run_speed_max       6608      6608        0    845       12.8
      inf_velo      12457     12457        0   2526       20.3

Overall: 24,846 / 129,29

In [12]:
# ============================================================
# APPLY OPTION B — NaN only stale outliers
# ============================================================
# A stale outlier is a stat value that is:
#   1. Outside ± OUTLIER_STD_DEV std devs from its commitment group mean
#   2. AND older than STALE_MONTHS months before graduation
#
# Uses STALE_MONTHS, OUTLIER_STD_DEV, STAT_DATE_PAIRS from the analysis cell above.
# Operates on inf_model_recent (inf_modeling stays untouched).

total_nand = 0
cleaning_log = []

for stat_col, date_col in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    stat_nand = 0

    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = inf_model_recent["committment_group"] == group
        vals = inf_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std  = vals.std()

        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue

        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale   = inf_model_recent.loc[mask, months_col] > STALE_MONTHS

        to_nan = is_outlier & is_stale
        n = to_nan.sum()

        if n > 0:
            # NaN the stat value and its date
            inf_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
            inf_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
            stat_nand += n

    if stat_nand > 0:
        cleaning_log.append({"stat": stat_col, "values_removed": stat_nand})
        total_nand += stat_nand

# --- Report ---
print(f"Option B applied: NaN'd {total_nand} stale outlier values")
print(f"  (±{OUTLIER_STD_DEV} std dev + >{STALE_MONTHS} months before graduation)\n")

if cleaning_log:
    log_df = pd.DataFrame(cleaning_log)
    print(log_df.to_string(index=False))

# --- Verify: no more stale outliers remain ---
remaining = 0
for stat_col, _ in STAT_DATE_PAIRS:
    months_col = f"{stat_col}__months_before_grad"
    for group in ["P4", "Non P4 D1", "Non D1"]:
        mask = inf_model_recent["committment_group"] == group
        vals = inf_model_recent.loc[mask, stat_col]
        mean = vals.mean()
        std  = vals.std()
        if pd.isna(mean) or pd.isna(std) or std == 0:
            continue
        is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
        is_stale   = inf_model_recent.loc[mask, months_col] > STALE_MONTHS
        remaining += (is_outlier & is_stale).sum()

print(f"\nVerification — stale outliers remaining: {remaining}")
print(f"inf_model_recent shape: {inf_model_recent.shape}")

Option B applied: NaN'd 115 stale outlier values
  (±2 std dev + >24 months before graduation)

          stat  values_removed
 exit_velo_max              32
 exit_velo_avg               5
  distance_max               2
hand_speed_max               1
 bat_speed_max               2
    hard_hit_p               4
    sixty_time              25
      inf_velo              44

Verification — stale outliers remaining: 11
inf_model_recent shape: (15451, 51)


In [ ]:
# ============================================================
# MULTI-PASS STALE OUTLIER CLEANING
# ============================================================
# Single pass leaves residual outliers because removing values shifts
# group means/stds, exposing new outliers. Loop until convergence.

MAX_PASSES = 5
total_removed = 0

for pass_num in range(1, MAX_PASSES + 1):
    pass_removed = 0
    for stat_col, date_col in STAT_DATE_PAIRS:
        months_col = f"{stat_col}__months_before_grad"
        for group in ["P4", "Non P4 D1", "Non D1"]:
            mask = inf_model_recent["committment_group"] == group
            vals = inf_model_recent.loc[mask, stat_col]
            mean = vals.mean()
            std = vals.std()
            if pd.isna(mean) or pd.isna(std) or std == 0:
                continue
            is_outlier = (vals < mean - OUTLIER_STD_DEV * std) | (vals > mean + OUTLIER_STD_DEV * std)
            is_stale = inf_model_recent.loc[mask, months_col] > STALE_MONTHS
            to_nan = is_outlier & is_stale
            n = to_nan.sum()
            if n > 0:
                inf_model_recent.loc[to_nan[to_nan].index, stat_col] = np.nan
                inf_model_recent.loc[to_nan[to_nan].index, date_col] = np.nan
                pass_removed += n

    total_removed += pass_removed
    print(f"Pass {pass_num}: removed {pass_removed} stale outlier values")
    if pass_removed == 0:
        print("Converged — no more stale outliers found.")
        break

print(f"\nTotal removed across all passes: {total_removed}")
print(f"inf_model_recent shape: {inf_model_recent.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

FEATURES = [
    "height", "weight",
    "exit_velo_max", "distance_max", 
    "bat_speed_max",
    "sixty_time",
    "inf_velo",
]

X = inf_model_recent[FEATURES]
y = inf_model_recent["d1_or_not"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = Pipeline([
    ("impute", KNNImputer(n_neighbors=10)),
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred), "\n")

coefs = pd.Series(pipe["lr"].coef_[0], index=FEATURES).sort_values()
print(coefs)

              precision    recall  f1-score   support

      Non D1       0.85      0.65      0.74      2273
          D1       0.41      0.69      0.52       818

    accuracy                           0.66      3091
   macro avg       0.63      0.67      0.63      3091
weighted avg       0.74      0.66      0.68      3091

[[1467  806]
 [ 250  568]] 

sixty_time      -0.416968
weight          -0.151033
bat_speed_max    0.041148
exit_velo_max    0.067585
distance_max     0.313986
height           0.383476
inf_velo         0.564259
dtype: float64


In [15]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# ============================================================
# XGBoost — 7 production features + Calibration
# ============================================================
FEATURES_XGB = [
    "height", "weight",
    "exit_velo_max", "distance_max",
    "sixty_time",
    "inf_velo",
    "bat_speed_max",
]

X = inf_model_recent[FEATURES_XGB]
y = inf_model_recent["d1_or_not"]

neg, pos = (y == 0).sum(), (y == 1).sum()
print(f"Class balance: {neg} Non D1, {pos} D1  (ratio {neg/pos:.2f}:1)\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)[:, 1]

print("XGBoost — 7 Features (holdout)")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred), "\n")

auc = roc_auc_score(y_test, y_proba)
print(f"AUC-ROC: {auc:.4f}")
print(f"  -> {auc:.0%} of the time, a random D1 player scores higher than a random Non-D1 player\n")

importances = pd.Series(xgb.feature_importances_, index=FEATURES_XGB).sort_values(ascending=False)
print(importances)

# --- Stratified 5-Fold CV ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    xgb, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\n\nStratified 5-Fold CV:")
print("=" * 55)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    train_scores = cv_results[f"train_{metric}"]
    test_scores = cv_results[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {train_scores.mean():.3f} (+/- {train_scores.std():.3f})  |  test {test_scores.mean():.3f} (+/- {test_scores.std():.3f})")

print(f"\n  Overfit gap (AUC): {cv_results['train_roc_auc'].mean() - cv_results['test_roc_auc'].mean():.3f}")

# --- Compare with LogReg baseline ---
xgb_report = classification_report(y_test, y_pred, target_names=["Non D1", "D1"], output_dict=True)
print(f"\n{'=' * 55}")
print("COMPARISON: LogReg (7 feat) vs XGBoost (7 feat)")
print(f"{'=' * 55}")
print(f"  LogReg baseline:  D1 precision=0.41  D1 recall=0.69  D1 F1=0.52  accuracy=0.66")
print(f"  XGBoost 7-feat:   D1 precision={xgb_report['D1']['precision']:.2f}  "
      f"D1 recall={xgb_report['D1']['recall']:.2f}  "
      f"D1 F1={xgb_report['D1']['f1-score']:.2f}  "
      f"accuracy={xgb_report['accuracy']:.2f}  "
      f"AUC={auc:.3f}")

# ============================================================
# Calibrated XGBoost — isotonic calibration
# ============================================================
xgb_calibrated = CalibratedClassifierCV(xgb, method="isotonic", cv=5)
xgb_calibrated.fit(X_train, y_train)

y_proba_cal = xgb_calibrated.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.45).astype(int)

print("\n\nCalibrated XGBoost — 7 Features (threshold=0.45)")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred_cal), "\n")

fraction_raw, mean_raw = calibration_curve(y_test, y_proba, n_bins=10)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)

print("Calibration comparison (raw vs calibrated):")
print("=" * 75)
print(f"  {'--- RAW ---':^28}   |   {'--- CALIBRATED ---':^28}")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}   |   {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
print(f"  {'-'*28}   |   {'-'*28}")

max_rows = max(len(mean_raw), len(mean_cal))
for i in range(max_rows):
    if i < len(mean_raw):
        r_gap = mean_raw[i] - fraction_raw[i]
        raw_str = f"  {mean_raw[i]:>10.2f}  {fraction_raw[i]:>8.2f}  {r_gap:>+8.2f}"
    else:
        raw_str = f"  {'':>28}"
    if i < len(mean_cal):
        c_gap = mean_cal[i] - fraction_cal[i]
        cal_str = f"{mean_cal[i]:>10.2f}  {fraction_cal[i]:>8.2f}  {c_gap:>+8.2f}"
        flag = "  <- off" if abs(c_gap) > 0.10 else ""
        cal_str += flag
    else:
        cal_str = f"{'':>28}"
    print(f"{raw_str}   |   {cal_str}")

brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)
print(f"\nBrier score:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}  |  improvement = {brier_raw - brier_cal:.4f}")

max_gap_raw = max(abs(mean_raw[i] - fraction_raw[i]) for i in range(len(mean_raw)))
max_gap_cal = max(abs(mean_cal[i] - fraction_cal[i]) for i in range(len(mean_cal)))
print(f"Max calibration gap:  raw = {max_gap_raw:.2f}  |  calibrated = {max_gap_cal:.2f}")
print(f"Worst-case Non-D1 PCI error:  raw = {max_gap_raw * 25:.1f} pts  |  calibrated = {max_gap_cal * 25:.1f} pts")

auc_cal = roc_auc_score(y_test, y_proba_cal)
print(f"AUC-ROC (calibrated): {auc_cal:.4f}")

Class balance: 11363 Non D1, 4088 D1  (ratio 2.78:1)

XGBoost — 7 Features (holdout)
              precision    recall  f1-score   support

      Non D1       0.87      0.66      0.75      2273
          D1       0.44      0.73      0.54       818

    accuracy                           0.68      3091
   macro avg       0.65      0.69      0.65      3091
weighted avg       0.76      0.68      0.70      3091

[[1505  768]
 [ 224  594]] 

AUC-ROC: 0.7759
  -> 78% of the time, a random D1 player scores higher than a random Non-D1 player

inf_velo         0.425010
sixty_time       0.145612
distance_max     0.129972
height           0.104937
exit_velo_max    0.098188
weight           0.051542
bat_speed_max    0.044738
dtype: float32


Stratified 5-Fold CV:
       AUC-ROC:  train 0.802 (+/- 0.003)  |  test 0.779 (+/- 0.011)
      accuracy:  train 0.693 (+/- 0.004)  |  test 0.681 (+/- 0.009)
            f1:  train 0.572 (+/- 0.004)  |  test 0.554 (+/- 0.013)
     precision:  train 0.453 (+/- 

In [16]:
CORE_7 = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "inf_velo", "bat_speed_max"]

inf_eng = inf_model_recent[CORE_7 + ["d1_or_not"]].copy()

# --- Interaction features ---
# Defensive tools: high arm velo + fast 60 = elite INF defender
inf_eng["arm_speed_ratio"] = inf_eng["inf_velo"] / inf_eng["sixty_time"]

# Raw two-way tools: arm strength x bat power
inf_eng["velo_x_exit"] = inf_eng["inf_velo"] * inf_eng["exit_velo_max"]

# Power tools: bat speed x exit velo = swing-to-contact transfer
inf_eng["bat_x_exit"] = inf_eng["bat_speed_max"] * inf_eng["exit_velo_max"]

# Distance x arm velo: overall power + arm combo
inf_eng["dist_x_arm"] = inf_eng["distance_max"] * inf_eng["inf_velo"]

# Body mass index proxy
inf_eng["bmi_proxy"] = inf_eng["weight"] / (inf_eng["height"] ** 2) * 703

# Speed-adjusted power: high exit velo relative to how fast you run
inf_eng["exit_per_sixty"] = inf_eng["exit_velo_max"] / inf_eng["sixty_time"]

# --- Check separation ---
ENG_FEATURES = [c for c in inf_eng.columns if c not in ["d1_or_not"]]

print(f"inf_eng: {inf_eng.shape[0]} rows, {len(ENG_FEATURES)} features")
print(f"  Core: {len(CORE_7)}  |  Engineered: {len(ENG_FEATURES) - len(CORE_7)}\n")

print("Cohen's d (sorted by |d|):")
print("=" * 55)
effects = []
for feat in ENG_FEATURES:
    d1 = inf_eng.loc[inf_eng["d1_or_not"] == 1, feat].dropna()
    nd1 = inf_eng.loc[inf_eng["d1_or_not"] == 0, feat].dropna()
    pooled = np.sqrt((nd1.std()**2 + d1.std()**2) / 2)
    d = (d1.mean() - nd1.mean()) / pooled if pooled > 0 else 0
    effects.append((feat, d))
for feat, d in sorted(effects, key=lambda x: abs(x[1]), reverse=True):
    tag = " *" if feat not in CORE_7 else ""
    print(f"  {feat:>20}  d = {d:+.3f}{tag}")

print("\n* = engineered feature")

# Correlation between engineered features and their parents
print("\nEngineered feature correlations with parents:")
print("=" * 55)
eng_pairs = [
    ("arm_speed_ratio", ["inf_velo", "sixty_time"]),
    ("velo_x_exit",     ["inf_velo", "exit_velo_max"]),
    ("bat_x_exit",      ["bat_speed_max", "exit_velo_max"]),
    ("dist_x_arm",      ["distance_max", "inf_velo"]),
    ("exit_per_sixty",  ["exit_velo_max", "sixty_time"]),
    ("bmi_proxy",       ["height", "weight"]),
]
for eng, parents in eng_pairs:
    corrs = [inf_eng[eng].corr(inf_eng[p]) for p in parents]
    corr_str = ", ".join(f"{p}={c:.2f}" for p, c in zip(parents, corrs))
    print(f"  {eng:>20} -> {corr_str}")

inf_eng: 15451 rows, 13 features
  Core: 7  |  Engineered: 6

Cohen's d (sorted by |d|):
       arm_speed_ratio  d = +1.051 *
           velo_x_exit  d = +1.029 *
            dist_x_arm  d = +1.026 *
              inf_velo  d = +0.956
        exit_per_sixty  d = +0.955 *
          distance_max  d = +0.781
            sixty_time  d = -0.774
         exit_velo_max  d = +0.736
            bat_x_exit  d = +0.715 *
         bat_speed_max  d = +0.567
                height  d = +0.376
                weight  d = +0.209
             bmi_proxy  d = +0.002 *

* = engineered feature

Engineered feature correlations with parents:
       arm_speed_ratio -> inf_velo=0.91, sixty_time=-0.80
           velo_x_exit -> inf_velo=0.87, exit_velo_max=0.84
            bat_x_exit -> bat_speed_max=0.94, exit_velo_max=0.89
            dist_x_arm -> distance_max=0.92, inf_velo=0.75
        exit_per_sixty -> exit_velo_max=0.86, sixty_time=-0.75
             bmi_proxy -> height=0.11, weight=0.84


In [17]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

# ============================================================
# XGBoost — Engineered Features + Calibration
# ============================================================
FEATURES_ENG = [c for c in inf_eng.columns if c != "d1_or_not"]

X = inf_eng[FEATURES_ENG]
y = inf_eng["d1_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_eng = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=3.0,
    min_child_weight=5,
    scale_pos_weight=neg / pos,
    eval_metric="logloss",
    random_state=42,
    enable_categorical=False,
)

xgb_eng.fit(X_train, y_train)
y_pred = xgb_eng.predict(X_test)
y_proba = xgb_eng.predict_proba(X_test)[:, 1]

print(f"XGBoost — {len(FEATURES_ENG)} Features (7 core + {len(FEATURES_ENG)-7} engineered)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=["Non D1", "D1"]))
print(confusion_matrix(y_test, y_pred))

auc_eng = roc_auc_score(y_test, y_proba)
print(f"\nAUC-ROC: {auc_eng:.4f}")

importances = pd.Series(xgb_eng.feature_importances_, index=FEATURES_ENG).sort_values(ascending=False)
print(f"\nFeature importance:")
print(importances.to_string())

# --- Stratified 5-Fold CV ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(
    xgb_eng, X, y, cv=skf,
    scoring=["roc_auc", "f1", "precision", "recall", "accuracy"],
    return_train_score=True,
)

print(f"\nStratified 5-Fold CV:")
print("=" * 60)
for metric in ["roc_auc", "accuracy", "f1", "precision", "recall"]:
    tr = cv[f"train_{metric}"]
    te = cv[f"test_{metric}"]
    label = "AUC-ROC" if metric == "roc_auc" else metric
    print(f"  {label:>12}:  train {tr.mean():.3f} (+/- {tr.std():.3f})  |  test {te.mean():.3f} (+/- {te.std():.3f})")
print(f"\n  Overfit gap (AUC): {cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean():.3f}")

# --- Calibration ---
xgb_eng_cal = CalibratedClassifierCV(xgb_eng, method="isotonic", cv=5)
xgb_eng_cal.fit(X_train, y_train)
y_proba_cal = xgb_eng_cal.predict_proba(X_test)[:, 1]
y_pred_cal = (y_proba_cal >= 0.45).astype(int)

fraction_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)
brier_raw = brier_score_loss(y_test, y_proba)
brier_cal = brier_score_loss(y_test, y_proba_cal)

print(f"\nCalibrated model (threshold=0.45):")
print("=" * 60)
print(classification_report(y_test, y_pred_cal, target_names=["Non D1", "D1"]))
print(f"Brier:  raw = {brier_raw:.4f}  |  calibrated = {brier_cal:.4f}")
print(f"\nCalibration curve (calibrated):")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

auc_eng_cal = roc_auc_score(y_test, y_proba_cal)
print(f"\nAUC-ROC (calibrated): {auc_eng_cal:.4f}")

# --- Comparison vs base 7-feature model ---
print(f"\n{'=' * 60}")
print(f"COMPARISON: 7-feature base vs {len(FEATURES_ENG)}-feature engineered")
print(f"{'=' * 60}")
print(f"  7-feat base:      AUC={auc:.4f}  Brier(cal)={brier_cal:.4f}")
print(f"  {len(FEATURES_ENG)}-feat eng:      AUC={auc_eng:.4f}  Brier(cal)={brier_cal:.4f}")
print(f"  AUC delta:        {auc_eng - auc:+.4f}")

XGBoost — 13 Features (7 core + 6 engineered)
              precision    recall  f1-score   support

      Non D1       0.88      0.66      0.76      2273
          D1       0.44      0.74      0.55       818

    accuracy                           0.68      3091
   macro avg       0.66      0.70      0.65      3091
weighted avg       0.76      0.68      0.70      3091

[[1507  766]
 [ 212  606]]

AUC-ROC: 0.7781

Feature importance:
velo_x_exit        0.266688
arm_speed_ratio    0.170698
dist_x_arm         0.119587
inf_velo           0.080292
height             0.067949
sixty_time         0.053244
distance_max       0.044609
exit_per_sixty     0.038633
exit_velo_max      0.036980
weight             0.036791
bmi_proxy          0.032484
bat_x_exit         0.031935
bat_speed_max      0.020112

Stratified 5-Fold CV:
       AUC-ROC:  train 0.806 (+/- 0.002)  |  test 0.780 (+/- 0.011)
      accuracy:  train 0.695 (+/- 0.005)  |  test 0.681 (+/- 0.006)
            f1:  train 0.578 (+/- 0.003

In [18]:
import joblib
import json
import os
from datetime import datetime

# ============================================================
# Save calibrated XGBoost to production model directory
# ============================================================
THRESHOLD = 0.45
VERSION = f"version_{datetime.now().strftime('%m%d%Y')}"
SAVE_DIR = os.path.abspath(os.path.join(
    os.getcwd(), "..", "..", "..", "models", "models_inf", "models_d1_or_not_inf", VERSION
))
os.makedirs(SAVE_DIR, exist_ok=True)

# --- Train final model on full training set ---
FEATURES_XGB = ["height", "weight", "exit_velo_max", "distance_max", "sixty_time", "inf_velo", "bat_speed_max"]

X = inf_model_recent[FEATURES_XGB]
y = inf_model_recent["d1_or_not"]
neg, pos = (y == 0).sum(), (y == 1).sum()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_final = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=3.0, min_child_weight=5,
    scale_pos_weight=neg / pos, eval_metric="logloss",
    random_state=42, enable_categorical=False,
)
xgb_final.fit(X_train, y_train)

xgb_final_cal = CalibratedClassifierCV(xgb_final, method="isotonic", cv=5)
xgb_final_cal.fit(X_train, y_train)

# --- Eval metrics on holdout ---
y_proba_test = xgb_final_cal.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= THRESHOLD).astype(int)

auc = roc_auc_score(y_test, y_proba_test)
brier = brier_score_loss(y_test, y_proba_test)
report = classification_report(y_test, y_pred_test, target_names=["Non D1", "D1"], output_dict=True)
fraction_cal, mean_cal = calibration_curve(y_test, y_proba_test, n_bins=10)
max_cal_gap = max(abs(m - f) for m, f in zip(mean_cal, fraction_cal))

# --- Save model ---
joblib.dump(xgb_final_cal, os.path.join(SAVE_DIR, "calibrated_xgb_model.pkl"))
print(f"Saved: calibrated_xgb_model.pkl")

# --- Save model config ---
model_config = {
    "model_version": f"inf_d1_xgb_cal_{VERSION}",
    "creation_date": datetime.now().isoformat(),
    "model_type": "calibrated_xgboost_outfielder_d1",
    "calibration_method": "isotonic",
    "threshold": THRESHOLD,
    "features": FEATURES_XGB,
    "hyperparameters": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.01,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 1.0,
        "reg_lambda": 3.0,
        "min_child_weight": 5,
        "scale_pos_weight": round(neg / pos, 4),
    },
    "performance_metrics": {
        "auc_roc": round(auc, 4),
        "brier_score": round(brier, 4),
        "max_calibration_gap": round(max_cal_gap, 4),
        "d1_precision": round(report["D1"]["precision"], 4),
        "d1_recall": round(report["D1"]["recall"], 4),
        "d1_f1": round(report["D1"]["f1-score"], 4),
        "accuracy": round(report["accuracy"], 4),
        "threshold_used": THRESHOLD,
    },
    "dataset_info": {
        "total_samples": len(inf_model_recent),
        "train_samples": len(X_train),
        "test_samples": len(X_test),
        "d1_rate": round(pos / (neg + pos), 4),
        "stale_cleaning": "Option B (stale outliers only, ±2 std + >24mo)",
    },
    "calibration_curve": {
        "predicted": [round(float(m), 4) for m in mean_cal],
        "actual": [round(float(f), 4) for f in fraction_cal],
    },
}

with open(os.path.join(SAVE_DIR, "model_config.json"), "w") as f:
    json.dump(model_config, f, indent=2)
print(f"Saved: model_config.json")

# --- Save feature metadata ---
feature_metadata = {
    "features": FEATURES_XGB,
    "required_columns": FEATURES_XGB,
    "notes": "XGBoost handles NaN natively. No imputation or scaling needed.",
}

with open(os.path.join(SAVE_DIR, "feature_metadata.json"), "w") as f:
    json.dump(feature_metadata, f, indent=2)
print(f"Saved: feature_metadata.json")

# --- Print summary ---
print(f"\nModel saved to: {SAVE_DIR}")
print(f"\n{'=' * 60}")
print(f"PRODUCTION MODEL METRICS (threshold={THRESHOLD})")
print(f"{'=' * 60}")
print(f"  AUC-ROC:           {auc:.4f}")
print(f"  Brier score:       {brier:.4f}")
print(f"  Max cal gap:       {max_cal_gap:.4f}")
print(f"  D1 Precision:      {report['D1']['precision']:.4f}")
print(f"  D1 Recall:         {report['D1']['recall']:.4f}")
print(f"  D1 F1:             {report['D1']['f1-score']:.4f}")
print(f"  Accuracy:          {report['accuracy']:.4f}")
print(f"  Features:          {len(FEATURES_XGB)} ({', '.join(FEATURES_XGB)})")
print(f"  Threshold:         {THRESHOLD}")
print(f"  Training samples:  {len(X_train)}")
print(f"  Test samples:      {len(X_test)}")
print(f"  D1 base rate:      {pos/(neg+pos):.1%}")

print(f"\nCalibration curve:")
print(f"  {'Predicted':>10}  {'Actual':>8}  {'Gap':>8}")
for pred, actual in zip(mean_cal, fraction_cal):
    gap = pred - actual
    flag = "  <- off" if abs(gap) > 0.10 else ""
    print(f"  {pred:>10.2f}  {actual:>8.2f}  {gap:>+8.2f}{flag}")

# Files saved
print(f"\nFiles in {VERSION}/:")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<35} {size:>8,} bytes")

Saved: calibrated_xgb_model.pkl
Saved: model_config.json
Saved: feature_metadata.json

Model saved to: /Users/ryankolodziejczyk/Documents/AI Baseball Recruitment/code/backend/ml/models/models_inf/models_d1_or_not_inf/version_04212026

PRODUCTION MODEL METRICS (threshold=0.45)
  AUC-ROC:           0.7753
  Brier score:       0.1574
  Max cal gap:       0.0492
  D1 Precision:      0.5627
  D1 Recall:         0.4279
  D1 F1:             0.4861
  Accuracy:          0.7606
  Features:          7 (height, weight, exit_velo_max, distance_max, sixty_time, inf_velo, bat_speed_max)
  Threshold:         0.45
  Training samples:  12360
  Test samples:      3091
  D1 base rate:      26.5%

Calibration curve:
   Predicted    Actual       Gap
        0.04      0.04     +0.00
        0.16      0.19     -0.03
        0.25      0.25     +0.00
        0.34      0.34     +0.00
        0.45      0.42     +0.03
        0.55      0.54     +0.02
        0.65      0.65     +0.00
        0.75      0.79     -0.0